### 01 - Imports

In [1]:
import os
import json
import random
import torch
import pandas as pd
import gradio as gr
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor
from qwen_vl_utils import process_vision_info

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

BASE_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
MERGED_A_DIR  = "../checkpoints/qwen2vl_2b_tuningA_merged"
MERGED_B_DIR  = "../checkpoints/qwen2vl_2b_tuningB_merged"
ADAPTER_A_DIR = "../checkpoints/qwen2vl_2b_tuningA/final"
ADAPTER_B_DIR = "../checkpoints/qwen2vl_2b_tuningB"
RESULTS_DIR = "../results"

SYSTEM_PROMPT = (
    "You are a VLM specialized in scientific diagram understanding. "
    "Answer multiple-choice questions about diagrams by selecting the correct option. "
    "Respond ONLY with the letter and answer text in this exact format: X) answer text. "
    "Example: C) stem"
)

print("✓ Config ready")

/home/matty/miniconda3/envs/ML/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Device: cuda
✓ Config ready


### 02 - Load all 3 models (merged checkpoints from 08_attention_maps)

In [2]:
from transformers import BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

def gpu_status():
    if not torch.cuda.is_available():
        print("No CUDA GPU available")
        return
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB | Free: {total - reserved:.2f} GB")

def clear_gpu():
    for var in ["model", "processor", "models", "merged", "base"]:
        if var in globals():
            del globals()[var]
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    gpu_status()

MODEL_ADAPTERS = {
    "Base (zero-shot)":              None,
    "Tuning A (image+Q)":            ADAPTER_A_DIR,
    "Tuning B (image+Q+annotation)": ADAPTER_B_DIR,
}

_loaded_name  = None
_loaded_model = None
_loaded_proc  = None

def get_model(model_name):
    global _loaded_name, _loaded_model, _loaded_proc
    if model_name == _loaded_name:
        return _loaded_model, _loaded_proc

    if _loaded_model is not None:
        del _loaded_model, _loaded_proc
        clear_gpu()

    print(f"Loading {model_name}...")
    base = Qwen2VLForConditionalGeneration.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb_config, torch_dtype=torch.bfloat16, device_map=device,
    )
    adapter_dir = MODEL_ADAPTERS[model_name]
    if adapter_dir is not None:
        base = PeftModel.from_pretrained(base, adapter_dir)  # NOT merged
    _loaded_proc = Qwen2VLProcessor.from_pretrained(BASE_MODEL_ID)
    base.eval()
    _loaded_model = base
    _loaded_name = model_name
    print(f"✓ {model_name} loaded")
    return _loaded_model, _loaded_proc

MODEL_PATHS = MODEL_ADAPTERS  # keep this name for the rest of the notebook (used in the loop over models)
print("✓ Lazy loader ready")

✓ Lazy loader ready


### 02b - Annotation context builder (used silently for Tuning B only)

In [3]:
AI2D_DIR = "../ai2d"

def build_annotation_context(image_name):
    ann_path = f"{AI2D_DIR}/annotations/{image_name}.json"
    if not os.path.exists(ann_path):
        return ""
    with open(ann_path) as f:
        ann = json.load(f)

    texts         = ann.get("text", {})
    relationships = ann.get("relationships", {})

    label_map = {}
    for tid, t in texts.items():
        val  = t.get("value", "").strip()
        repl = t.get("replacementText", "").strip()
        if val:
            label_map[tid] = f"{val}({repl})" if repl else val

    connections = []
    title       = None
    for rel in relationships.values():
        cat        = rel.get("category", "")
        origin_str = label_map.get(rel.get("origin", ""))
        dest_str   = label_map.get(rel.get("destination", ""))
        if cat == "imageTitle" and origin_str:
            title = origin_str
        elif cat == "intraObjectLabel":
            if origin_str and dest_str:
                connections.append(f"{origin_str} labels {dest_str}")
            elif origin_str:
                connections.append(f"{origin_str} → visual region")
        elif cat == "intraObjectLinkage" and origin_str:
            connections.append(f"{origin_str} → visual region")

    context = ""
    if title:
        context += f"Diagram title: {title}\n"
    if label_map:
        context += f"Diagram labels: {', '.join(label_map.values())}\n"
    if connections:
        context += "Connections:\n" + "\n".join(f"  {c}" for c in connections) + "\n"
    return context

print("✓ Annotation builder ready")

✓ Annotation builder ready


### 03 - Inference function

In [4]:
def run_inference(model_name, image, question_text, image_name=None):
    model, processor = get_model(model_name)
    image = image.convert("RGB")

    prompt_text = question_text
    if model_name == "Tuning B (image+Q+annotation)" and image_name:
        ann_context = build_annotation_context(image_name)
        if ann_context:
            prompt_text = f"{ann_context}{question_text}"

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": prompt_text},
        ]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=16, do_sample=False)
    generated_ids = [out[len(inp):] for inp, out in zip(inputs["input_ids"], generated_ids)]
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

print("✓ Inference function ready")

✓ Inference function ready


### 04 - Load test-set examples to use as defaults

In [5]:
df_base = pd.read_csv(f"{RESULTS_DIR}/predictions/qwen2vl_2b_base.csv")
df_A    = pd.read_csv(f"{RESULTS_DIR}/qwen2vl_2b_tuningA.csv")
df_B    = pd.read_csv(f"{RESULTS_DIR}/qwen2vl_2b_tuningB.csv")

merged = (
    df_base[["question_id", "correct"]].rename(columns={"correct": "correct_base"})
    .merge(df_A[["question_id", "correct"]].rename(columns={"correct": "correct_A"}), on="question_id")
    .merge(df_B[["question_id", "correct"]].rename(columns={"correct": "correct_B"}), on="question_id")
)

with open("../processed_data/splits/test_vA.json") as f:
    test_A = json.load(f)
qid_to_sample = {s["question_id"]: s for s in test_A}

def pick_n(mask, n=2):
    sub = merged[mask]
    return sub["question_id"].head(n).tolist()

qids_both_win = pick_n((~merged.correct_base) & (merged.correct_A)  & (merged.correct_B))
qids_A_only   = pick_n((~merged.correct_base) & (merged.correct_A)  & (~merged.correct_B))
qids_B_only   = pick_n((~merged.correct_base) & (~merged.correct_A) & (merged.correct_B))

categories = [
    ("Base fails, A & B correct", qids_both_win),
    ("Only A correct",            qids_A_only),
    ("Only B correct",            qids_B_only),
]

selected_qids = []
for label, qids in categories:
    if len(qids) < 2:
        print(f"⚠ Only found {len(qids)}/2 example(s) for: {label}")
    selected_qids += qids

missing = 6 - len(selected_qids)
if missing:
    print(f"⚠ Filling {missing} slot(s) with random examples instead")
    fallback = merged[~merged["question_id"].isin(selected_qids)]["question_id"].sample(missing, random_state=42)
    selected_qids += fallback.tolist()

default_pool = [qid_to_sample[q] for q in selected_qids]

# What the UI sees — unchanged
DEFAULT_EXAMPLES = [
    [s["image_path"], s["user_text"], s["answer"], s["image_name"]] for s in default_pool
]

# Backend-only: map image_path -> image_name, so run_all_models can silently
# fetch the right annotation for Tuning B without any new UI fields
image_path_to_name = {s["image_path"]: s["image_name"] for s in default_pool}

for label, qids in categories:
    for q in qids:
        print(f"{label}: {q}")
print(f"✓ Loaded {len(DEFAULT_EXAMPLES)} default examples")

Base fails, A & B correct: 3089.png-1
Base fails, A & B correct: 3647.png-1
Only A correct: 3089.png-0
Only A correct: 2083.png-5
Only B correct: 1218.png-0
Only B correct: 3715.png-2
✓ Loaded 6 default examples


In [5]:
with open("../processed_data/splits/test_vA.json") as f:
    test_A = json.load(f)
 
random.seed(42)
abc_examples  = [s for s in test_A if s["abc_label"]]
desc_examples = [s for s in test_A if not s["abc_label"]]
default_pool  = random.sample(abc_examples, min(3, len(abc_examples))) + \
                random.sample(desc_examples, min(3, len(desc_examples)))
 
DEFAULT_EXAMPLES = [
    [s["image_path"], s["user_text"], s["answer"]] for s in default_pool
]
print(f"✓ Loaded {len(DEFAULT_EXAMPLES)} default examples")

✓ Loaded 6 default examples


### 05 - Compare-all-models helper

In [6]:
def run_all_models(image, question_text, image_name):
    if image is None or not question_text:
        return "Please provide an image and a question.", "", ""
    outputs = []
    for name in MODEL_PATHS:
        try:
            pred = run_inference(name, image, question_text, image_name=image_name)
        except Exception as e:
            pred = f"[error: {e}]"
        outputs.append(pred)
    return outputs[0], outputs[1], outputs[2]

### 06 - Gradio interface

In [7]:
with gr.Blocks(title="AI2D VQA — Base vs Tuning A vs Tuning B") as demo:
    gr.Markdown("## AI2D Diagram QA — Model Comparison\nPick or upload a diagram, enter a question, and compare all 3 model outputs side by side.")
 
    with gr.Row():
        with gr.Column(scale=1):
            image_in = gr.Image(type="pil", label="Diagram")
            question_in = gr.Textbox(
                label="Question (include A) B) C) D) options)",
                lines=6,
                placeholder="Question: What does the arrow point to?\nA) leaf  B) root  C) stem  D) flower",
            )
            gt_display = gr.Textbox(label="Ground truth (from example, if selected)", interactive=False)
            image_name_state = gr.State(value=None)
            run_btn = gr.Button("Run all 3 models", variant="primary")
 
        with gr.Column(scale=1):
            out_base = gr.Textbox(label="Base (zero-shot)")
            out_A    = gr.Textbox(label="Tuning A (image+Q)")
            out_B    = gr.Textbox(label="Tuning B (image+Q+annotation)")
 
    gr.Examples(
        examples=DEFAULT_EXAMPLES,
        inputs=[image_in, question_in, gt_display, image_name_state],
        label="Sample test-set questions (click to load)",
    )
 
    run_btn.click(
        fn=run_all_models,
        inputs=[image_in, question_in, image_name_state],
        outputs=[out_base, out_A, out_B],
    )
 
print("✓ Interface built")

✓ Interface built


### 07 - Launch

In [9]:
import os
print(os.getcwd())

/home/matty/Documents/UE_DataScience/Sem_II/Machine Learning/VLM_FInal_Project/FINAL/AI2D-VQA-Project/mattychan/notebooks


In [ ]:
import os

AI2D_IMAGES_DIR = os.path.abspath("../ai2d/images")

demo.launch(share=True, debug=False, allowed_paths=[AI2D_IMAGES_DIR])

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://24432b5f26eda83c3f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


id=f63b2799aa56aaca730d14481151d8fece64f56cf9777e41ee5fdea320aeaefc, environment=client, type=example, file_name=style.css
id=f63b2799aa56aaca730d14481151d8fece64f56cf9777e41ee5fdea320aeaefc, environment=client, type=example, file_name=index.js
id=f63b2799aa56aaca730d14481151d8fece64f56cf9777e41ee5fdea320aeaefc, environment=client, type=example, file_name=svelte_runtime_entry.js
id=f63b2799aa56aaca730d14481151d8fece64f56cf9777e41ee5fdea320aeaefc, environment=client, type=example, file_name=style.css
id=f63b2799aa56aaca730d14481151d8fece64f56cf9777e41ee5fdea320aeaefc, environment=client, type=example, file_name=index.js
id=f63b2799aa56aaca730d14481151d8fece64f56cf9777e41ee5fdea320aeaefc, environment=client, type=example, file_name=svelte_runtime_entry.js
Loading Base (zero-shot)...


Loading weights: 100%|██████████| 729/729 [00:05<00:00, 132.68it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✓ Base (zero-shot) loaded
Allocated: 0.01 GB | Reserved: 0.06 GB | Free: 6.03 GB
Loading Tuning A (image+Q)...


Loading weights: 100%|██████████| 729/729 [00:05<00:00, 133.22it/s]


...

✓ Base (zero-shot) loaded
Allocated: 0.01 GB | Reserved: 0.06 GB | Free: 6.03 GB
Loading Tuning A (image+Q)...


Loading weights: 100%|██████████| 729/729 [00:04<00:00, 150.83it/s]


✓ Tuning A (image+Q) loaded
Allocated: 0.01 GB | Reserved: 0.06 GB | Free: 6.03 GB
Loading Tuning B (image+Q+annotation)...


Loading weights: 100%|██████████| 729/729 [00:04<00:00, 155.08it/s]


✓ Tuning B (image+Q+annotation) loaded
